# Generate MultiSocial Arabic Dataset (Colab)

This notebook creates `multisocial_micro_ar.csv` from your own uploaded CSV file using a single generation provider: NVIDIA NIM (OpenAI-compatible).

## Required packages

- openai
- pandas
- tqdm
- scikit-learn
- python-dotenv

## Expected input

Upload a CSV using this schema:
`["label", "tweet"]`

## Before running

Set the NVIDIA API key in Colab environment variables (never hardcode keys in notebook cells):

- `NVIDIA_API_KEY`

## Execution

This notebook processes all sampled rows with NVIDIA NIM (`meta/llama-4-maverick-17b-128e-instruct`).


In [ ]:
# Install dependencies (Colab)
!pip -q install openai pandas tqdm scikit-learn python-dotenv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import logging
import os
import time
from pathlib import Path
from typing import List, Tuple

import openai
import pandas as pd
from dotenv import load_dotenv
from openai import APIConnectionError, APIStatusError, APITimeoutError, InternalServerError, RateLimitError
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# -----------------------------
# Global constants
# -----------------------------
RANDOM_SEED = 42
HUMAN_SAMPLE_SIZE = 2000
MODEL_NAME = 'meta/llama-4-maverick-17b-128e-instruct'
OUTPUT_CSV_PATH = '/content/drive/MyDrive/multisocial_outputs/multisocial_micro_ar.csv'
INPUT_CSV_PATH_DEFAULT = '/content/drive/MyDrive/multisocial_outputs/Arabic_tweets_df.csv'
MAX_RETRIES = 3
SUCCESS_CALL_SLEEP_SECONDS = 0.5
NVIDIA_BASE_URL = 'https://integrate.api.nvidia.com/v1'
SOURCE_NAME = 'ajgt'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s'
 )
logger = logging.getLogger('multisocial_ar')

from google.colab import userdata
nvidia_api_key = userdata.get('NVIDIA_API_KEY')
if not nvidia_api_key:
    raise EnvironmentError(
        'Missing NVIDIA_API_KEY environment variable. Set it before running this notebook.'
    )

client = openai.OpenAI(
    api_key=nvidia_api_key,
    base_url=NVIDIA_BASE_URL
)

logger.info('NVIDIA client initialized.')
logger.info('Base URL: %s', NVIDIA_BASE_URL)
logger.info('Model: %s', MODEL_NAME)

In [ ]:
# Optional tuning cell for quotas and speed
MODEL_NAME = 'meta/llama-4-maverick-17b-128e-instruct'
MAX_RETRIES = 3
SUCCESS_CALL_SLEEP_SECONDS = 0.5
HUMAN_SAMPLE_SIZE = 2000

print(f'NVIDIA model: {MODEL_NAME}')
print(f'Retries: {MAX_RETRIES} | success sleep: {SUCCESS_CALL_SLEEP_SECONDS}s')
print(f'Sample size: {HUMAN_SAMPLE_SIZE}')

NVIDIA model: meta/llama-4-maverick-17b-128e-instruct
Retries: 3 | success sleep: 0.5s
Sample size: 2000


In [ ]:
def _is_retriable_error(exc: Exception) -> bool:
    """Detect transient errors that should be retried."""
    if isinstance(exc, (RateLimitError, APIConnectionError, APITimeoutError, InternalServerError)):
        return True

    if isinstance(exc, APIStatusError):
        status_code = getattr(exc, 'status_code', None)
        if status_code == 429:
            return True
        if isinstance(status_code, int) and status_code >= 500:
            return True

    status_code = getattr(exc, 'status_code', None)
    if status_code == 429:
        return True

    message = str(exc).lower()
    return '429' in message or 'rate' in message or 'timeout' in message or 'connection' in message


def paraphrase_arabic(text: str, client, model, max_retries: int = 3) -> tuple[bool, str]:
    """Generate one Arabic paraphrase with exponential backoff retries."""
    prompt = (
        'Paraphrase the following Arabic social media post. Keep the exact same informal tone, '
        'length, hashtags, and emojis. IMPORTANT: Output the paraphrased text ONLY in Arabic. '
        'Do not translate it to English or any other language. Do not add conversational fillers. '
        'Just output the paraphrased text. Text: '
        f'{text}'
    )

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.7,
                max_tokens=512,
            )
            paraphrased_text = (response.choices[0].message.content or '').strip()
            if paraphrased_text:
                return True, paraphrased_text
            logger.warning('Empty response at attempt %s/%s', attempt, max_retries)
        except Exception as exc:
            retriable = _is_retriable_error(exc)
            logger.warning(
                'Paraphrasing failed at attempt %s/%s (retriable=%s): %s',
                attempt,
                max_retries,
                retriable,
                exc,
            )
            if not retriable:
                return False, ''

        if attempt < max_retries:
            sleep_seconds = 2 ** attempt
            logger.info('Sleeping %s second(s) before retry...', sleep_seconds)
            time.sleep(sleep_seconds)

    return False, ''


def load_human_data(csv_path: str = INPUT_CSV_PATH_DEFAULT, n_samples: int = 2000, seed: int = 42) -> pd.DataFrame:
    """
    Load and sample human-written Arabic social texts from CSV.

    Required columns:
        - label
        - tweet
    """
    csv_file = Path(csv_path)
    if not csv_file.exists():
        raise FileNotFoundError(f'Input CSV not found: {csv_path}')

    raw_df = pd.read_csv(csv_file)
    required_columns = {'label', 'tweet'}
    missing_columns = required_columns.difference(raw_df.columns)
    if missing_columns:
        raise ValueError(
            f"Input CSV must contain columns {sorted(required_columns)}. Missing: {sorted(missing_columns)}"
        )

    text_series = raw_df['tweet'].astype('string').fillna('').str.strip()
    text_series = text_series[text_series != '']

    if len(text_series) < n_samples:
        raise ValueError(
            f'Requested n_samples={n_samples}, but only {len(text_series)} non-empty rows are available.'
        )

    sampled_texts = text_series.sample(n=n_samples, random_state=seed, replace=False).tolist()
    logger.info('Input label distribution: %s', raw_df['label'].value_counts(dropna=False).to_dict())

    human_df = pd.DataFrame({
        'text': pd.Series(sampled_texts, dtype='string').fillna(''),
        'label': 0,
        'multi_label': 'human',
        'source': SOURCE_NAME,
    })

    logger.info('Human data prepared: %s rows', len(human_df))
    return human_df


def generate_machine_data(human_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generate machine-written counterparts and keep only successful pairs."""
    successful_humans: List[dict] = []
    generated_rows: List[dict] = []
    attempted_count = 0
    success_count = 0
    dropped_count = 0

    logger.info('Starting NVIDIA generation for %s rows...', len(human_df))

    try:
        for text in tqdm(human_df['text'].tolist(), desc='Generating (nvidia)', unit='post'):
            attempted_count += 1
            original_text = str(text)
            success, paraphrased_text = paraphrase_arabic(
                text=original_text,
                client=client,
                model=MODEL_NAME,
                max_retries=MAX_RETRIES,
            )

            if success:
                successful_humans.append({
                    'text': original_text,
                    'label': 0,
                    'multi_label': 'human',
                    'source': SOURCE_NAME,
                })
                generated_rows.append({
                    'text': paraphrased_text,
                    'label': 1,
                    'multi_label': 'llama-4-maverick-17b',
                    'source': SOURCE_NAME,
                })
                success_count += 1
            else:
                dropped_count += 1

            time.sleep(SUCCESS_CALL_SLEEP_SECONDS)
    finally:
        logger.info(
            'Generation progress: attempted=%s success=%s dropped=%s',
            attempted_count,
            success_count,
            dropped_count,
        )

    if success_count == 0:
        raise RuntimeError(
            'No successful generations were produced. Check NVIDIA_API_KEY/quota and retry settings.'
        )

    machine_df = pd.DataFrame(generated_rows)
    balanced_human_df = pd.DataFrame(successful_humans[: len(machine_df)])

    logger.info('Successful pairs kept: %s', len(machine_df))
    return balanced_human_df, machine_df


def add_split_with_stratification(df: pd.DataFrame) -> pd.DataFrame:
    """Create a stratified 80/20 train/test split by label."""
    _, test_idx = train_test_split(
        df.index,
        test_size=0.2,
        random_state=RANDOM_SEED,
        stratify=df['label'],
        shuffle=True,
    )

    df = df.copy()
    df['split'] = 'train'
    df.loc[test_idx, 'split'] = 'test'
    return df


def format_and_save(human_df: pd.DataFrame, machine_df: pd.DataFrame, output_path: str) -> pd.DataFrame:
    """Concatenate, enrich, validate, and save final dataset."""
    if len(human_df) != len(machine_df):
        raise RuntimeError('Human and machine row counts must be equal.')

    combined_df = pd.concat([human_df, machine_df], ignore_index=True)
    combined_df['text'] = combined_df['text'].astype(str)
    combined_df['language'] = 'ar'
    combined_df['length'] = combined_df['text'].apply(lambda x: len(x.split()))
    combined_df['potential_noise'] = 0
    combined_df = add_split_with_stratification(combined_df)

    final_columns = [
        'text',
        'label',
        'multi_label',
        'split',
        'language',
        'length',
        'source',
        'potential_noise',
    ]
    final_df = combined_df[final_columns]

    if final_df.isna().any().any():
        raise RuntimeError('Missing values detected in final dataset.')

    label_counts = final_df['label'].value_counts().to_dict()
    if label_counts.get(0, 0) != label_counts.get(1, 0):
        raise RuntimeError(f'Class imbalance detected: {label_counts}')

    final_df.to_csv(output_path, index=False)
    logger.info('Saved dataset to %s', output_path)
    logger.info('Final rows: %s | Human: %s | Machine: %s', len(final_df), len(human_df), len(machine_df))
    return final_df


def main(input_csv_path: str) -> pd.DataFrame:
    """Run the full Arabic generation pipeline."""
    logger.info('Pipeline started (NVIDIA, ar).')
    human_df = load_human_data(csv_path=input_csv_path, n_samples=HUMAN_SAMPLE_SIZE, seed=RANDOM_SEED)
    paired_human_df, machine_df = generate_machine_data(human_df)
    final_df = format_and_save(paired_human_df, machine_df, output_path=OUTPUT_CSV_PATH)
    logger.info('Pipeline completed (NVIDIA, ar).')
    return final_df

In [ ]:
# Run NVIDIA pipeline on sampled Arabic rows
input_csv_path = INPUT_CSV_PATH_DEFAULT
if not os.path.exists(input_csv_path):
    raise FileNotFoundError(f'Could not find the file at {input_csv_path}. Please check the path.')

print(f'Using input CSV: {input_csv_path}')
final_df = main(input_csv_path=input_csv_path)
print('Completed NVIDIA ar run')
display(final_df.head())

Using input CSV: /content/drive/MyDrive/multisocial_outputs/Arabic_tweets_df.csv


Generating (nvidia): 100%|██████████| 2000/2000 [43:34<00:00,  1.31s/post]

Completed NVIDIA ar run


,text,label,multi_label,split,language,length,source,potential_noise
0,قوتنا بعد الله دائما أنتم🌹💙 الف مبرووك 💙 الحمد...,0,human,train,ar,14,ajgt,0
1,‌صباح_الخير صباح ھادئ 🍃 يعآنق السماء ب امنيات ...,0,human,test,ar,13,ajgt,0
2,صباح الورد 🍁,0,human,train,ar,3,ajgt,0
3,وأحب أسافر مع سحاب تعلى 💙,0,human,train,ar,6,ajgt,0
4,"مرة سألهم شخص ليش چذي مغلدمين مادمتم ""على الصر...",0,human,train,ar,20,ajgt,0


In [ ]:
# Quick verification checks (dynamic row count after failed generations are dropped)
expected_columns = [
    'text',
    'label',
    'multi_label',
    'split',
    'language',
    'length',
    'source',
    'potential_noise',
]
assert list(final_df.columns) == expected_columns, 'Column order does not match required schema.'

label_counts = final_df['label'].value_counts().to_dict()
assert set(label_counts.keys()) == {0, 1}, f'Unexpected labels found: {label_counts}'
assert label_counts[0] == label_counts[1], f'Class imbalance detected: {label_counts}'

split_counts = final_df.groupby(['split', 'label']).size().unstack(fill_value=0)
display(split_counts)
display(final_df['split'].value_counts())

assert (final_df['language'] == 'ar').all(), 'language must be ar for all rows.'
assert (final_df['source'] == SOURCE_NAME).all(), f'source must be {SOURCE_NAME} for all rows.'
assert (final_df['potential_noise'] == 0).all(), 'potential_noise must be 0 for all rows.'
assert not final_df.isna().any().any(), 'No missing values are allowed in final_df.'

print(f'Final paired rows per class: {label_counts[0]}')
print(f'Total rows: {len(final_df)}')

label,0,1
split,,
test,400,400
train,1600,1600


,count
split,
train,3200
test,800


Final paired rows per class: 2000
Total rows: 4000


In [ ]:
import shutil

drive_output_dir = Path('/content/drive/MyDrive/multisocial_outputs')
drive_output_dir.mkdir(parents=True, exist_ok=True)
drive_output_path = drive_output_dir / OUTPUT_CSV_PATH

if 'final_df' in globals():
    final_df.to_csv(drive_output_path, index=False)
elif Path(OUTPUT_CSV_PATH).exists():
    shutil.copy2(OUTPUT_CSV_PATH, drive_output_path)
else:
    raise FileNotFoundError(
        f'Could not find {OUTPUT_CSV_PATH}. Run the execution cell first to generate the dataset.'
    )

print(f'Saved output to: {drive_output_path}')
print(f'File size: {drive_output_path.stat().st_size / 1024:.2f} KB')

Saved output to: /content/drive/MyDrive/multisocial_outputs/multisocial_micro_ar.csv
File size: 806.11 KB
